In [1]:


import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")



In [2]:

#  LOAD DATASET

CSV_PATH = "air_quality_dataset_tunis.csv"
df = pd.read_csv(CSV_PATH,sep=';')
print(df.columns.tolist())

print(f"\n Dataset loaded: {len(df)} samples")
print(df["label"].value_counts().to_string())

print(df.head(10))


['mode', 'temp', 'humidity', 'mq2', 'mq135', 'label']

 Dataset loaded: 1740 samples
label
GOOD                540
MEDIUM              480
POOR                360
DANGER_GAS          180
DANGER_POLLUTION    180
    mode  temp  humidity   mq2  mq135             label
0  CLASS  20.2      59.2   843   1095              GOOD
1    LAB  25.5      64.1  1090   1004            MEDIUM
2    LAB  20.0      64.8   361    320              GOOD
3  AMPHI  12.7      52.1  3764   1240        DANGER_GAS
4  CLASS  26.8      71.9  1722   1699            MEDIUM
5    LAB  18.6      67.4  1230   3700  DANGER_POLLUTION
6  AMPHI  29.5      85.0  2999   2315              POOR
7  CLASS  25.6      65.3  1423   1289            MEDIUM
8    LAB  29.5      81.5  1988   2064              POOR
9    LAB  24.4      62.3   997   1076            MEDIUM


In [3]:

#  FEATURE ENGINEERING


mode_map = {"CLASS": 0, "AMPHI": 1, "LAB": 2}
df["mode_enc"] = df["mode"].map(mode_map)


df["heat_index"] = df["temp"] * (1 + 0.01 * df["humidity"])  # simplified
df["gas_ratio"]  = df["mq2"] / (df["mq135"] + 1)

FEATURES = ["mode_enc", "temp", "humidity", "mq2", "mq135", "heat_index", "gas_ratio"]
TARGET   = "label"

X = df[FEATURES].values
y = df[TARGET].values


le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"\n Classes: {list(le.classes_)}")



 Classes: ['DANGER_GAS', 'DANGER_POLLUTION', 'GOOD', 'MEDIUM', 'POOR']


In [4]:

#  TRAIN / TEST 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"\n[3] Train: {len(X_train)} | Test: {len(X_test)}")




[3] Train: 1392 | Test: 348


In [5]:

#  SCALE 


scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)



In [6]:

#  TRAIN RANDOM FOREST  

print("\n Training Random Forest ...")
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

cv_rf = cross_val_score(rf, X, y_enc, cv=StratifiedKFold(5), scoring="accuracy")
print(f"   Accuracy : {acc_rf:.4f}")
print(f"   CV mean  : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")




 Training Random Forest ...
   Accuracy : 0.9914
   CV mean  : 0.9845 ± 0.0076


In [7]:

#  TRAIN DECISION TREE  

print("\n Training Decision Tree (for MCU export) ...")
dt = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)

cv_dt = cross_val_score(dt, X, y_enc, cv=StratifiedKFold(5), scoring="accuracy")
print(f"   Accuracy : {acc_dt:.4f}")
print(f"   CV mean  : {cv_dt.mean():.4f} ± {cv_dt.std():.4f}")




 Training Decision Tree (for MCU export) ...
   Accuracy : 0.9626
   CV mean  : 0.9776 ± 0.0028


In [8]:
# SAVE MODELS

joblib.dump(rf,      "rf_air_quality.pkl")
joblib.dump(dt,      "dt_air_quality.pkl")
joblib.dump(scaler,  "scaler.pkl")
joblib.dump(le,      "label_encoder.pkl")
print("\n Models saved: rf_air_quality.pkl, dt_air_quality.pkl")




 Models saved: rf_air_quality.pkl, dt_air_quality.pkl


In [9]:
# CLASSIFICATION REPORT

label_names = list(le.classes_)

report_rf = classification_report(y_test, y_pred_rf, target_names=label_names)
report_dt = classification_report(y_test, y_pred_dt, target_names=label_names)

print("\n Random Forest Report:")
print(report_rf)
print(" Decision Tree Report:")
print(report_dt)




 Random Forest Report:
                  precision    recall  f1-score   support

      DANGER_GAS       1.00      1.00      1.00        36
DANGER_POLLUTION       1.00      1.00      1.00        36
            GOOD       1.00      1.00      1.00       108
          MEDIUM       0.99      0.98      0.98        96
            POOR       0.97      0.99      0.98        72

        accuracy                           0.99       348
       macro avg       0.99      0.99      0.99       348
    weighted avg       0.99      0.99      0.99       348

 Decision Tree Report:
                  precision    recall  f1-score   support

      DANGER_GAS       1.00      1.00      1.00        36
DANGER_POLLUTION       1.00      1.00      1.00        36
            GOOD       0.94      0.99      0.96       108
          MEDIUM       0.96      0.92      0.94        96
            POOR       0.97      0.94      0.96        72

        accuracy                           0.96       348
       macro avg    

In [10]:

#   CONFUSION MATRIX PLOTS

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_p, title in [
    (axes[0], y_pred_rf, f"Random Forest (acc={acc_rf:.3f})"),
    (axes[1], y_pred_dt, f"Decision Tree (acc={acc_dt:.3f})")
]:
    cm = confusion_matrix(y_test, y_p)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
    ax.set_title(title, fontsize=13, fontweight="bold")

plt.suptitle("Air Quality Classifier — Confusion Matrices (Tunis Indoor)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
print("\n confusion_matrices.png saved")




 confusion_matrices.png saved


In [11]:

#   FEATURE IMPORTANCE PLOT

feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig2, ax2 = plt.subplots(figsize=(8, 4))
feat_imp.plot(kind="barh", color="steelblue", ax=ax2)
ax2.set_title("Feature Importances — Random Forest", fontweight="bold")
ax2.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
print("[10] feature_importance.png saved")



[10] feature_importance.png saved


In [12]:

#   EXPORT DECISION TREE → ESP32 C HEADER


from sklearn.tree import _tree

def tree_to_c(tree, feature_names, class_names, func_name="predictAirQuality"):
    """
    Converts a sklearn DecisionTree to a C function (char* return).
    Safe for ESP32 — no malloc, no STL, pure if/else.
    """
    tree_ = tree.tree_
    fn    = feature_names

    lines = []
    lines.append("")
    lines.append('#ifndef AIR_QUALITY_MODEL_H')
    lines.append('#define AIR_QUALITY_MODEL_H')
    lines.append("")
    lines.append("/* Call this function from your loop():")
    lines.append("   int modeEnc = (mode == \"CLASS\") ? 0 : (mode == \"AMPHI\") ? 1 : 2;")
    lines.append("   float heatIndex = temp * (1.0f + 0.01f * hum);")
    lines.append("   float gasRatio  = (float)mq2 / (mq135 + 1.0f);")
    lines.append("   String label = predictAirQuality(modeEnc, temp, hum, mq2, mq135,")
    lines.append("                                    heatIndex, gasRatio);")
    lines.append("*/")
    lines.append("")
    lines.append(f"inline String {func_name}(")
    lines.append("    int   mode_enc,")
    lines.append("    float temp,")
    lines.append("    float humidity,")
    lines.append("    int   mq2,")
    lines.append("    int   mq135,")
    lines.append("    float heat_index,")
    lines.append("    float gas_ratio)")
    lines.append("{")

    def recurse(node, depth):
        indent = "    " * (depth + 1)
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            feat  = fn[tree_.feature[node]]
            thr   = tree_.threshold[node]
            # pick appropriate type
            is_int = feat in ["mode_enc", "mq2", "mq135"]
            thr_s  = f"{int(thr)}" if is_int else f"{thr:.4f}f"
            lines.append(f"{indent}if ({feat} <= {thr_s}) {{")
            recurse(tree_.children_left[node],  depth + 1)
            lines.append(f"{indent}}} else {{")
            recurse(tree_.children_right[node], depth + 1)
            lines.append(f"{indent}}}")
        else:
            cls_idx = int(np.argmax(tree_.value[node]))
            cls_name = class_names[cls_idx]
            lines.append(f'{indent}return "{cls_name}";')

    recurse(0, 0)
    lines.append("}")
    lines.append("")
    lines.append("#endif // AIR_QUALITY_MODEL_H")
    return "\n".join(lines)

c_code = tree_to_c(dt, FEATURES, label_names)

with open("model_esp32.h", "w") as f:
    f.write(c_code)

print(" model_esp32.h written (C header for ESP32)")



 model_esp32.h written (C header for ESP32)


In [13]:

#   WRITE TEXT REPORT

with open("model_report.txt", "w") as f:
    f.write("AIR QUALITY ML MODEL REPORT\n")
    f.write(f"Dataset: {len(df)} samples\n")
    f.write(f"Features: {FEATURES}\n")
    f.write(f"Classes : {label_names}\n\n")
    f.write(f"Random Forest  — Test Accuracy: {acc_rf:.4f}\n")
    f.write(f"Random Forest  — CV  5-fold   : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}\n\n")
    f.write(f"Decision Tree  — Test Accuracy: {acc_dt:.4f}\n")
    f.write(f"Decision Tree  — CV  5-fold   : {cv_dt.mean():.4f} ± {cv_dt.std():.4f}\n\n")
    f.write("--- Random Forest Classification Report ---\n")
    f.write(report_rf + "\n")
    f.write("--- Decision Tree Classification Report ---\n")
    f.write(report_dt + "\n")
    f.write("\n--- Decision Tree Rules ---\n")
    f.write(export_text(dt, feature_names=FEATURES, max_depth=5))

print(" model_report.txt written")

print("\n  Training complete. Output files:")
print("   - rf_air_quality.pkl       (Random Forest model)")
print("   - dt_air_quality.pkl       (Decision Tree model)")
print("   - scaler.pkl               (StandardScaler)")
print("   - label_encoder.pkl        (LabelEncoder)")
print("   - model_esp32.h            (ESP32 C header)")
print("   - confusion_matrices.png")

 model_report.txt written

  Training complete. Output files:
   - rf_air_quality.pkl       (Random Forest model)
   - dt_air_quality.pkl       (Decision Tree model)
   - scaler.pkl               (StandardScaler)
   - label_encoder.pkl        (LabelEncoder)
   - model_esp32.h            (ESP32 C header)
   - confusion_matrices.png
